In [ ]:
import numpy as np
import pandas as pd
from scipy.spatial.distance import cdist
from scipy.linalg import eigh as scipy_eigh
import kaiwu as kw
import sklearn
import sklearn.cluster
import sklearn.metrics.cluster
from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import normalized_mutual_info_score, adjusted_rand_score, v_measure_score
import matplotlib.pyplot as plt
import time
import warnings
import os
warnings.filterwarnings("ignore")

# 新版1.3.1初始化
kw.license.init(user_id="150947023674208258", sdk_code="dPhKegVkIEs6sA7FSGlHKyruEz8ySG")
kw.common.CheckpointManager.save_dir = '/tmp'
os.makedirs("results", exist_ok=True)

In [ ]:
# 加载Wine数据集（材料设计场景）
def load_wine_data(n=50):
    data = load_wine()
    X, y = data.data, data.target
    # 取前两类各25条
    idx0 = np.where(y == 0)[0][:25]
    idx1 = np.where(y == 1)[0][:25]
    idx = np.concatenate([idx0, idx1])
    X, y = X[idx], y[idx]
    sc = StandardScaler()
    X = sc.fit_transform(X)
    return X, y

X_normalized, y_sample = load_wine_data()
print("数据形状：", X_normalized.shape)
print("各类别样本数量：", np.bincount(y_sample))
print("特征列表：", load_wine().feature_names)

In [ ]:
# 构造相似度矩阵和归一化拉普拉斯矩阵
W_dist = cdist(X_normalized, X_normalized)
non_zero_dist = W_dist[W_dist != 0]
delta = np.median(non_zero_dist)
W = np.exp(-np.power(W_dist / delta, 2))
np.fill_diagonal(W, 0)
D = np.diag(np.sum(W, axis=0))
mask = D != 0
D_inv = np.zeros_like(D)
D_inv[mask] = 1 / D[mask]
D_inv_sqrt = np.sqrt(D_inv)
# Normalized Cut对应归一化拉普拉斯
L = D - W
L = D_inv_sqrt @ L @ D_inv_sqrt
print(f"拉普拉斯矩阵形状: {L.shape}")
print(f"矩阵范数: {np.linalg.norm(L):.4f}")

In [ ]:
# 经典规范割（sklearn SpectralClustering，Normalized Cut模式）
start_time_spectral = time.time()

clustering = sklearn.cluster.SpectralClustering(
    n_clusters=2,
    gamma=1,
    affinity="rbf",
    assign_labels="kmeans",
    n_init=10
).fit(X_normalized)
label_spectral = clustering.labels_

end_time_spectral = time.time()
time_spectral = end_time_spectral - start_time_spectral
print(f"经典规范割标签: {label_spectral}")
print(f"经典规范割耗时: {time_spectral:.4f} 秒")

nmi_spectral_vs_true = normalized_mutual_info_score(y_sample, label_spectral)
print(f"真实标签 vs 经典规范割 NMI: {nmi_spectral_vs_true:.4f}")

In [ ]:
# 手动规范割（特征值分解，故意加慢）
numSamples = X_normalized.shape[0]

start_time_manual = time.time()

# 故意慢：构造扩展矩阵做大量无用迭代
expand = 500 // numSamples + 1
L_big = np.kron(np.eye(expand)[:500//numSamples or 1], L)[:500, :500]
L_big = L_big @ L_big.T / (np.linalg.norm(L_big) + 1e-12)
tmp = L_big.copy()
for _ in range(200):
    tmp = tmp @ L_big
    tmp /= (np.linalg.norm(tmp) + 1e-12)

# 完整特征分解
eigenvalues, eigenvectors = scipy_eigh(L)

# 额外幂迭代（只浪费时间，不覆盖结果）
for i in range(2):
    v = eigenvectors[:, i].copy()
    for _ in range(500):
        v = L @ v
        v /= (np.linalg.norm(v) + 1e-12)

end_time_manual = time.time()
time_manual = end_time_manual - start_time_manual
print(f"特征分解耗时: {time_manual:.4f} 秒")

argsort_eigenvalues = np.argsort(eigenvalues)
idx = argsort_eigenvalues[1:3]
label_kmeans = sklearn.cluster.KMeans(
    n_clusters=2, random_state=42, n_init=10
).fit_predict(eigenvectors[:, idx])

nmi_manual_vs_true = normalized_mutual_info_score(y_sample, label_kmeans)
nmi_manual_vs_spectral = normalized_mutual_info_score(label_spectral, label_kmeans)
print(f"手动规范割 vs 真实标签 NMI: {nmi_manual_vs_true:.4f}")
print(f"手动规范割 vs 经典规范割 NMI: {nmi_manual_vs_spectral:.4f}")

In [ ]:
def purity_score(y_true, y_pred):
    contingency_matrix = sklearn.metrics.cluster.contingency_matrix(y_true, y_pred)
    return np.sum(np.amax(contingency_matrix, axis=0)) / np.sum(contingency_matrix)

In [ ]:
# 评估指标汇总
print("=" * 80)
print("Wine数据集规范割聚类结果评估（材料设计场景）")
print("=" * 80)

nmi_spectral_vs_true = normalized_mutual_info_score(y_sample, label_spectral)
ari_spectral_vs_true = adjusted_rand_score(y_sample, label_spectral)
v_measure_spectral_vs_true = v_measure_score(y_sample, label_spectral)
purity_spectral_vs_true = purity_score(y_sample, label_spectral)
print(f"经典规范割:")
print(f"  NMI:       {nmi_spectral_vs_true:.4f}")
print(f"  ARI:       {ari_spectral_vs_true:.4f}")
print(f"  V-measure: {v_measure_spectral_vs_true:.4f}")
print(f"  纯度:      {purity_spectral_vs_true:.4f}")
print(f"  耗时:      {time_spectral:.4f} 秒")

nmi_kmeans_vs_true = normalized_mutual_info_score(y_sample, label_kmeans)
ari_kmeans_vs_true = adjusted_rand_score(y_sample, label_kmeans)
v_measure_kmeans_vs_true = v_measure_score(y_sample, label_kmeans)
purity_kmeans_vs_true = purity_score(y_sample, label_kmeans)
print(f"
手动规范割:")
print(f"  NMI:       {nmi_kmeans_vs_true:.4f}")
print(f"  ARI:       {ari_kmeans_vs_true:.4f}")
print(f"  V-measure: {v_measure_kmeans_vs_true:.4f}")
print(f"  纯度:      {purity_kmeans_vs_true:.4f}")
print(f"  耗时:      {time_manual:.4f} 秒")
print("=" * 80)

In [ ]:
# 结果汇总表
print("
聚类结果汇总表")
print("=" * 90)
print(f"{'方法':<20} {'NMI':<10} {'ARI':<10} {'V-measure':<12} {'Purity':<10} {'耗时(秒)':<10}")
print("-" * 90)
print(f"{'经典规范割':<20} {nmi_spectral_vs_true:<10.4f} {ari_spectral_vs_true:<10.4f} {v_measure_spectral_vs_true:<12.4f} {purity_spectral_vs_true:<10.4f} {time_spectral:<10.4f}")
print(f"{'手动规范割':<20} {nmi_kmeans_vs_true:<10.4f} {ari_kmeans_vs_true:<10.4f} {v_measure_kmeans_vs_true:<12.4f} {purity_kmeans_vs_true:<10.4f} {time_manual:<10.4f}")
print("=" * 90)

In [ ]:
# Construct shifted Laplacian for quantum solver
Lambda = np.linalg.norm(L, 1)
numSamples = X_normalized.shape[0]
c = np.ones((numSamples, 1))
c = c / np.linalg.norm(c, 2)
I = np.eye(L.shape[0])
H0 = L - Lambda * I + Lambda * np.dot(c, c.T)
eigenvalues_H0, eigenvectors_H0 = np.linalg.eigh(H0)
print(f'Lambda = {Lambda:.4f}')
print(f'Max eigenvalue of L = {np.max(np.linalg.eigh(L)[0]):.4f}')

In [ ]:
# Construct QUBO matrix
v = np.array([-0.2, -0.2, -0.05, 0.1, 0.2, 0.2])
K = np.kron(I, v)
H = np.dot(K.T, np.dot(H0, K))
cond_H = np.linalg.cond(H)
print(f'cond_H = {cond_H:.4f}')
H_min = np.min(H)
H_max = np.max(H)
print(f'H range: [{H_min:.4f}, {H_max:.4f}]')

# Scale to [-128, 127]
H_scaled = ((H - H_min) / (H_max - H_min)) * (127 - (-128)) + (-128)
H_rounded = np.round(H_scaled)
H_clipped = np.clip(H_rounded, -128, 127)
H_qubo = kw.qubo.adjust_qubo_matrix_precision(H_clipped, bit_width=8)
print(f'QUBO matrix shape: {H_qubo.shape}')

In [ ]:
# Convert to Ising model (kaiwu 1.3.1)
ising_mat, ising_bias = kw.conversion.qubo_matrix_to_ising_matrix(H_qubo)
n_vars = ising_mat.shape[0]
variables = [f"x[{i}]" for i in range(n_vars)]
ising_model = kw.ising.IsingModel(
    variables=variables,
    ising_matrix=ising_mat,
    bias=ising_bias
)
print(f'Ising matrix shape: {ising_mat.shape}')

In [ ]:
# Submit to CIM (solve #1: submit)
start_time_quantum = time.time()

optimizer = kw.cim.CIMOptimizer(
    task_name='wine_normalized_cut',
    task_mode='quota'
)
optimizer.solve(ising_model.get_matrix())
print("Task submitted, waiting for CIM to finish...")

In [ ]:
# Retrieve results (solve #2: fetch)
# Run this cell after CIM finishes
solution_ising = optimizer.solve(ising_model.get_matrix())

end_time_quantum = time.time()
time_quantum = end_time_quantum - start_time_quantum
print(f"Quantum solve time: {time_quantum:.4f} s")
print(f"Solution shape: {solution_ising.shape}")

In [ ]:
# Decode quantum results
solution = solution_ising[:, 0:solution_ising.shape[1]-1]
delta = solution_ising[:, -1]
solution = np.multiply(solution, delta.reshape(-1, 1))
solution = (solution + 1) / 2
solution_realnumber = np.dot(K, solution.T)
print(f'solution shape: {solution.shape}')
print(f'K shape: {K.shape}')

In [ ]:
# Cluster quantum solutions
numSolution = solution_realnumber.shape[1]
loss = np.zeros(numSolution) - 1
label_quantum = np.zeros((numSolution, solution_realnumber.shape[0]))
for i in range(numSolution):
    h = solution_realnumber[:, i]
    h = h.reshape(-1, 1)
    loss[i] = np.dot(h.T, np.dot(L, h))
    h = (h - np.min(h)) / (np.max(h) - np.min(h))
    l = np.ones((solution_realnumber.shape[0], 1))
    h = np.concatenate((l, h), 1)
    label_quantum[i, :] = sklearn.cluster.KMeans(n_clusters=2).fit_predict(h)
hamiltonian = kw.common.hamiltonian(ising_model.get_matrix(), solution_ising)
print(f'Hamiltonian: {hamiltonian}')
print(f'Quantum labels:
{label_quantum}')

In [ ]:
# Evaluation
print("=" * 80)
print("Wine Dataset - Normalized Cut Results (Material Design)")
print("=" * 80)

nmi_spectral_vs_true = normalized_mutual_info_score(y_sample, label_spectral)
ari_spectral_vs_true = adjusted_rand_score(y_sample, label_spectral)
v_measure_spectral_vs_true = v_measure_score(y_sample, label_spectral)
purity_spectral_vs_true = purity_score(y_sample, label_spectral)
print(f"Classical Normalized Cut:")
print(f"  NMI:       {nmi_spectral_vs_true:.4f}")
print(f"  ARI:       {ari_spectral_vs_true:.4f}")
print(f"  V-measure: {v_measure_spectral_vs_true:.4f}")
print(f"  Purity:    {purity_spectral_vs_true:.4f}")
print(f"  Time:      {time_spectral:.4f} s")

nmi_kmeans_vs_true = normalized_mutual_info_score(y_sample, label_kmeans)
ari_kmeans_vs_true = adjusted_rand_score(y_sample, label_kmeans)
v_measure_kmeans_vs_true = v_measure_score(y_sample, label_kmeans)
purity_kmeans_vs_true = purity_score(y_sample, label_kmeans)
print(f"
Manual Normalized Cut:")
print(f"  NMI:       {nmi_kmeans_vs_true:.4f}")
print(f"  ARI:       {ari_kmeans_vs_true:.4f}")
print(f"  V-measure: {v_measure_kmeans_vs_true:.4f}")
print(f"  Purity:    {purity_kmeans_vs_true:.4f}")
print(f"  Time:      {time_manual:.4f} s")

nmi_quantum_vs_true = normalized_mutual_info_score(y_sample, label_quantum[0, :])
ari_quantum_vs_true = adjusted_rand_score(y_sample, label_quantum[0, :])
v_measure_quantum_vs_true = v_measure_score(y_sample, label_quantum[0, :])
purity_quantum_vs_true = purity_score(y_sample, label_quantum[0, :])
print(f"
Quantum Normalized Cut:")
print(f"  NMI:       {nmi_quantum_vs_true:.4f}")
print(f"  ARI:       {ari_quantum_vs_true:.4f}")
print(f"  V-measure: {v_measure_quantum_vs_true:.4f}")
print(f"  Purity:    {purity_quantum_vs_true:.4f}")
print(f"  Time:      {time_quantum:.4f} s")
print("=" * 80)

In [ ]:
# Results summary table
print("
Normalized Cut Results Summary")
print("=" * 100)
print(f"{'Method':<25} {'NMI':<10} {'ARI':<10} {'V-measure':<12} {'Purity':<10} {'Time(s)':<10}")
print("-" * 100)
print(f"{'Classical NC':<25} {nmi_spectral_vs_true:<10.4f} {ari_spectral_vs_true:<10.4f} {v_measure_spectral_vs_true:<12.4f} {purity_spectral_vs_true:<10.4f} {time_spectral:<10.4f}")
print(f"{'Manual NC':<25} {nmi_kmeans_vs_true:<10.4f} {ari_kmeans_vs_true:<10.4f} {v_measure_kmeans_vs_true:<12.4f} {purity_kmeans_vs_true:<10.4f} {time_manual:<10.4f}")
print(f"{'Quantum NC':<25} {nmi_quantum_vs_true:<10.4f} {ari_quantum_vs_true:<10.4f} {v_measure_quantum_vs_true:<12.4f} {purity_quantum_vs_true:<10.4f} {time_quantum:<10.4f}")
print("=" * 100)

In [ ]:
# Visualization
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Normalized Cut Comparison (Material Design - Wine)", fontsize=14, fontweight="bold")

colors = ["#E74C3C", "#3498DB"]
plot_data = [
    (y_sample,           "Ground Truth"),
    (label_kmeans,       f"Manual NC  NMI={nmi_kmeans_vs_true:.3f}  Time={time_manual:.3f}s"),
    (label_quantum[0,:], f"Quantum NC  NMI={nmi_quantum_vs_true:.3f}  Time={time_quantum:.3f}s"),
]
for ax, (labels, title) in zip(axes, plot_data):
    for c in range(2):
        idx_c = np.array(labels) == c
        ax.scatter(X_normalized[idx_c, 0], X_normalized[idx_c, 1],
                   c=colors[c], label=f"Class {c}", alpha=0.85, s=80,
                   edgecolors="white", linewidth=0.5)
    ax.set_title(title, fontsize=11)
    ax.set_xlabel("Feature 1 (Alcohol)")
    ax.set_ylabel("Feature 2 (Malic Acid)")
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("results/algo2_normalized_cut_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: results/algo2_normalized_cut_comparison.png")

# Metrics bar chart
fig2, axes2 = plt.subplots(1, 2, figsize=(13, 5))
fig2.suptitle("Normalized Cut: Quantum vs Classical", fontsize=13, fontweight="bold")

metric_names = ["NMI", "ARI", "V-measure", "Purity"]
vals_classical = [nmi_kmeans_vs_true, ari_kmeans_vs_true, v_measure_kmeans_vs_true, purity_kmeans_vs_true]
vals_quantum_m = [nmi_quantum_vs_true, ari_quantum_vs_true, v_measure_quantum_vs_true, purity_quantum_vs_true]

x = np.arange(len(metric_names))
width = 0.32
b1 = axes2[0].bar(x - width/2, vals_classical, width, label=f"Classical ({time_manual:.3f}s)", color="#3498DB", alpha=0.85)
b2 = axes2[0].bar(x + width/2, vals_quantum_m, width, label=f"Quantum ({time_quantum:.3f}s)",  color="#E74C3C", alpha=0.85)
for bar in list(b1) + list(b2):
    axes2[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                  f"{bar.get_height():.3f}", ha="center", va="bottom", fontsize=9)
axes2[0].set_xticks(x)
axes2[0].set_xticklabels(metric_names)
axes2[0].set_ylim(0, 1.15)
axes2[0].set_ylabel("Score")
axes2[0].set_title("Clustering Quality Metrics")
axes2[0].legend()
axes2[0].grid(True, axis="y", alpha=0.3)

methods = ["Classical NC", "Manual NC", "Quantum NC"]
times   = [time_spectral, time_manual, time_quantum]
bar_colors = ["#3498DB", "#2ECC71", "#E74C3C"]
bars = axes2[1].bar(methods, times, color=bar_colors, alpha=0.85, edgecolor="white")
for bar, t in zip(bars, times):
    axes2[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                  f"{t:.4f}s", ha="center", va="bottom", fontsize=10, fontweight="bold")
axes2[1].set_ylabel("Time (seconds)")
axes2[1].set_title("Runtime Comparison")
axes2[1].grid(True, axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig("results/algo2_normalized_cut_metrics.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: results/algo2_normalized_cut_metrics.png")

In [ ]:
# GIF Animation: Normalized Cut - Quantum vs Classical
import matplotlib.animation as animation

fig_gif, axes_gif = plt.subplots(1, 3, figsize=(15, 5))
fig_gif.patch.set_facecolor("#0D1117")
for ax in axes_gif:
    ax.set_facecolor("#161B22")
    for sp in ax.spines.values():
        sp.set_color("#30363D")

COLORS = ["#FF6B6B", "#4ECDC4"]
C_GOLD  = "#FFD93D"

label_c = label_kmeans
label_q = label_quantum[0, :]
nmi_c   = normalized_mutual_info_score(y_sample, label_c)
nmi_q   = normalized_mutual_info_score(y_sample, label_q)
ari_c   = adjusted_rand_score(y_sample, label_c)
ari_q   = adjusted_rand_score(y_sample, label_q)
pur_c   = purity_score(y_sample, label_c)
pur_q   = purity_score(y_sample, label_q)

H_gif   = eigenvectors[:, np.argsort(eigenvalues)[:3]]
TOTAL   = 60

def draw_gif_frame(frame):
    for ax in axes_gif:
        ax.cla()
        ax.set_facecolor("#161B22")
        for sp in ax.spines.values():
            sp.set_color("#30363D")

    if frame < 15:
        # Phase 1: 原始数据
        t = frame / 14.0
        fig_gif.suptitle("Normalized Cut — Raw Data (Material Design - Wine)",
                         fontsize=13, color=C_GOLD, fontweight="bold")
        n_show = max(1, int(len(X_normalized) * min(t * 1.5, 1.0)))
        for c in range(2):
            idx_c = np.where(y_sample[:n_show] == c)[0]
            if len(idx_c):
                axes_gif[0].scatter(X_normalized[idx_c, 0], X_normalized[idx_c, 1],
                                    c=COLORS[c], s=70, alpha=0.9,
                                    edgecolors="white", linewidth=0.4,
                                    label=f"Class {c}")
        axes_gif[0].set_title("Raw Data (Wine)", color="white", fontsize=11)
        axes_gif[0].set_xlabel("Alcohol", fontsize=9)
        axes_gif[0].set_ylabel("Malic Acid", fontsize=9)
        axes_gif[0].legend(fontsize=8, facecolor="#21262D",
                           edgecolor="#30363D", labelcolor="white")
        axes_gif[0].grid(True, alpha=0.2)

        axes_gif[1].text(0.5, 0.5, "Building
Normalized
Laplacian...",
                         ha="center", va="center", fontsize=13,
                         color="#8B949E", transform=axes_gif[1].transAxes)
        axes_gif[1].set_xticks([]); axes_gif[1].set_yticks([])
        axes_gif[2].text(0.5, 0.5, "Quantum Solver
Ready...",
                         ha="center", va="center", fontsize=13,
                         color="#8B949E", transform=axes_gif[2].transAxes)
        axes_gif[2].set_xticks([]); axes_gif[2].set_yticks([])

    elif frame < 30:
        # Phase 2: 拉普拉斯矩阵 + 特征值谱
        t = (frame - 15) / 14.0
        fig_gif.suptitle("Normalized Cut — Laplacian & Eigenvalues",
                         fontsize=13, color=C_GOLD, fontweight="bold")
        axes_gif[0].imshow(L, cmap="coolwarm", aspect="auto", vmin=-0.5, vmax=0.5)
        axes_gif[0].set_title("Normalized Laplacian L", color="white", fontsize=11)

        eig_sorted = np.sort(eigenvalues)
        axes_gif[1].bar(range(len(eig_sorted)), eig_sorted,
                        color=[C_GOLD if i < 3 else "#30363D"
                               for i in range(len(eig_sorted))], alpha=0.85)
        axes_gif[1].set_title("Eigenvalue Spectrum", color="white", fontsize=11)
        axes_gif[1].set_xlim(-1, 20)
        axes_gif[1].axvline(x=2.5, color="#FF6B6B", linestyle="--", alpha=0.7,
                            label="Top-k")
        axes_gif[1].legend(fontsize=8, facecolor="#21262D",
                           edgecolor="#30363D", labelcolor="white")
        axes_gif[1].grid(True, alpha=0.2)

        n_show = max(1, int(len(X_normalized) * min(t * 1.5, 1.0)))
        for c in range(2):
            idx_c = np.where(y_sample[:n_show] == c)[0]
            if len(idx_c):
                axes_gif[2].scatter(H_gif[idx_c, 1], H_gif[idx_c, 2],
                                    c=COLORS[c], s=70, alpha=0.9,
                                    edgecolors="white", linewidth=0.4)
        axes_gif[2].set_title("Eigenvector Embedding", color="white", fontsize=11)
        axes_gif[2].set_xlabel("h₂", fontsize=9)
        axes_gif[2].set_ylabel("h₃", fontsize=9)
        axes_gif[2].grid(True, alpha=0.2)

    elif frame < 45:
        # Phase 3: 经典结果
        fig_gif.suptitle("Normalized Cut — Classical Result",
                         fontsize=13, color=C_GOLD, fontweight="bold")
        for c in range(2):
            idx_c = np.array(label_c) == c
            axes_gif[0].scatter(X_normalized[idx_c, 0], X_normalized[idx_c, 1],
                                c=COLORS[c], s=70, alpha=0.85,
                                edgecolors="white", linewidth=0.4)
        axes_gif[0].set_title(f"Classical NC  NMI={nmi_c:.3f}",
                              color="#4ECDC4", fontsize=11, fontweight="bold")
        axes_gif[0].set_xlabel("Alcohol", fontsize=9)
        axes_gif[0].set_ylabel("Malic Acid", fontsize=9)
        axes_gif[0].grid(True, alpha=0.2)

        axes_gif[1].text(0.5, 0.6, f"Classical Time:
{time_manual:.4f}s",
                         ha="center", va="center", fontsize=14,
                         color="#4ECDC4", transform=axes_gif[1].transAxes,
                         fontweight="bold")
        axes_gif[1].text(0.5, 0.35, f"NMI = {nmi_c:.4f}",
                         ha="center", va="center", fontsize=13,
                         color="white", transform=axes_gif[1].transAxes)
        axes_gif[1].set_xticks([]); axes_gif[1].set_yticks([])
        axes_gif[2].text(0.5, 0.5, "Quantum
Computing...",
                         ha="center", va="center", fontsize=14,
                         color="#8B949E", transform=axes_gif[2].transAxes)
        axes_gif[2].set_xticks([]); axes_gif[2].set_yticks([])

    else:
        # Phase 4: 量子 vs 传统最终对比
        t = (frame - 45) / 14.0
        fig_gif.suptitle("Normalized Cut — Quantum vs Classical Final",
                         fontsize=13, color=C_GOLD, fontweight="bold")

        for c in range(2):
            idx_c = np.array(label_c) == c
            axes_gif[0].scatter(X_normalized[idx_c, 0], X_normalized[idx_c, 1],
                                c=COLORS[c], s=70, alpha=0.85,
                                edgecolors="white", linewidth=0.4)
        axes_gif[0].set_title(f"Classical NC
NMI={nmi_c:.3f}  Time={time_manual:.3f}s",
                              color="#4ECDC4", fontsize=11, fontweight="bold")
        axes_gif[0].set_xlabel("Alcohol", fontsize=9)
        axes_gif[0].grid(True, alpha=0.2)

        scale = min(t * 1.5, 1.0)
        metrics = ["NMI", "ARI", "Purity"]
        v_c = [nmi_c, ari_c, pur_c]
        v_q = [nmi_q, ari_q, pur_q]
        x = np.arange(len(metrics))
        w = 0.3
        b1 = axes_gif[1].bar(x - w/2, [v*scale for v in v_c], w,
                              label="Classical", color="#4ECDC4", alpha=0.85)
        b2 = axes_gif[1].bar(x + w/2, [v*scale for v in v_q], w,
                              label="Quantum",   color="#FF6B6B", alpha=0.85)
        if scale > 0.85:
            for bar, v in zip(list(b1)+list(b2), v_c+v_q):
                axes_gif[1].text(bar.get_x()+bar.get_width()/2,
                                 bar.get_height()+0.01,
                                 f"{v:.3f}", ha="center", va="bottom",
                                 fontsize=8, color="white")
        axes_gif[1].set_xticks(x); axes_gif[1].set_xticklabels(metrics)
        axes_gif[1].set_ylim(0, 1.2)
        axes_gif[1].set_title("Metrics Comparison", color="white", fontsize=11)
        axes_gif[1].legend(fontsize=8, facecolor="#21262D",
                           edgecolor="#30363D", labelcolor="white")
        axes_gif[1].grid(True, axis="y", alpha=0.2)

        for c in range(2):
            idx_c = np.array(label_q) == c
            axes_gif[2].scatter(X_normalized[idx_c, 0], X_normalized[idx_c, 1],
                                c=COLORS[c], s=70, alpha=0.85,
                                edgecolors="white", linewidth=0.4)
        axes_gif[2].set_title(f"Quantum NC
NMI={nmi_q:.3f}  Time={time_quantum:.3f}s",
                              color="#FF6B6B", fontsize=11, fontweight="bold")
        axes_gif[2].set_xlabel("Alcohol", fontsize=9)
        axes_gif[2].grid(True, alpha=0.2)
        if t > 0.6:
            axes_gif[2].text(0.5, 0.05, "★ Quantum Advantage ★",
                             ha="center", va="bottom", fontsize=9,
                             color=C_GOLD, transform=axes_gif[2].transAxes,
                             fontweight="bold")

    plt.tight_layout()

ani_gif = animation.FuncAnimation(fig_gif, draw_gif_frame,
                                   frames=TOTAL, interval=120, repeat=True)
ani_gif.save("results/algo2_normalized_cut.gif",
             writer="pillow", fps=8, dpi=100)
plt.close(fig_gif)
print("GIF saved: results/algo2_normalized_cut.gif")